# Agentic Workflows with CrewAI

> **Focus Area:** Multi-agent collaboration
> **Topics:** Agent (role/goal/backstory); Task + context; Crew + Process (sequential, parallel, hierarchical)
> **Achievement:** Researcher, Writer, Reviewer টিম, local `llama3.1:8b` দিয়ে full report লেখে

Setup: `pip install crewai` (Python 3.10-3.13, venv repo-র বাইরে)। এই notebook-এর সব cell Ollama ছাড়াই চলে; শুধু শেষ cell-এ Ollama থাকলে crew সত্যিই চলে।

## 1. Agent: role, goal, backstory

তিনটা field মিলে agent-এর system prompt হয়। `role` সে কে, `goal` কী অর্জন করবে, `backstory` কেমন মানসিকতায় কাজ করবে। `LLM(model="ollama/...")` লিখলে CrewAI local Ollama server ব্যবহার করে; object বানাতে network লাগে না।

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM

# HIGHLIGHTS: llama3.1:8b কেন? Agent-কে বারবার format ধরে instruction মানতে হয়; 8B model এতে
# নির্ভরযোগ্য আর laptop-এ চলে। বিকল্প: mistral:7b (দ্রুত, format মাঝে মাঝে ভাঙে),
# llama3.2:3b (খুব হালকা, reviewer/manager-এ দুর্বল), qwen2.5:7b (structured output ভালো),
# llama3.1:70b (সেরা মান, ~40GB+ memory)।
llm = LLM(model="ollama/llama3.1:8b", base_url="http://localhost:11434", temperature=0.2)

researcher = Agent(
    role="Senior Research Specialist",
    goal="Collect accurate, well-organised facts about the topic.",
    backstory="You separate facts from speculation and flag anything uncertain.",
    llm=llm, allow_delegation=False)
writer = Agent(
    role="Technical Report Writer",
    goal="Turn research notes into a clear Markdown report.",
    backstory="You write for busy engineers: short sections, concrete examples.",
    llm=llm, allow_delegation=False)
reviewer = Agent(
    role="Critical Reviewer",
    goal="Fix gaps and return the final polished report.",
    backstory="You never invent facts; you cut claims the notes do not support.",
    llm=llm, allow_delegation=False)
print([a.role for a in (researcher, writer, reviewer)])

## 2. Reference table: কোন model কখন

| Model | Use case | Trade-off |
|---|---|---|
| llama3.1:8b | Default, balanced | ~8GB RAM |
| mistral:7b | দ্রুত লেখা | Format মাঝে মাঝে ভাঙে |
| llama3.2:3b | খুব হালকা machine | Reviewer/manager-এ দুর্বল |
| qwen2.5:7b | Structured output | ধীর হতে পারে |
| llama3.1:70b | সেরা মান | ~40GB+ memory |

## 3. Task এবং context

`context=[task]` হলো dependency: আগের task-এর output পরের task-এর prompt-এ যুক্ত হয়। `expected_output` বলে শেষ ফল কেমন হবে।

In [ ]:
topic = "Vector databases"

research = Task(description=f"Research: {topic}", expected_output="15 bullet facts", agent=researcher)
write = Task(description=f"Write a Markdown report on {topic}", expected_output="Full report",
             agent=writer, context=[research])
review = Task(description="Review and return the FINAL report", expected_output="Final Markdown",
              agent=reviewer, context=[write])

for t in (research, write, review):
    print(t.agent.role, "<- context:", [c.agent.role for c in (t.context or [])])

## 4. Sequential Crew

`Process.sequential` মানে task গুলো তালিকার ক্রমে চলে। `kickoff()` চালালে `result.raw` পাওয়া যায়, আর `result.tasks_output` এ প্রতিটা task-এর output।

In [ ]:
crew = Crew(agents=[researcher, writer, reviewer], tasks=[research, write, review],
            process=Process.sequential)
print(crew.process, len(crew.tasks), "tasks")

## 5. Parallel (async) tasks

`async_execution=True` দিলে task background-এ শুরু হয়। যে task এদের output চায় সে `context=[...]` দিয়ে অপেক্ষা করে। শেষ task async হতে পারে না। একটাই local Ollama হলে বাস্তবে speed-up কম, কারণ request গুলো একই model-এ লাইন ধরে।

In [ ]:
bg = Task(description=f"Background of {topic}", expected_output="10 facts", agent=researcher, async_execution=True)
trends = Task(description=f"Trends and risks of {topic}", expected_output="10 facts", agent=researcher, async_execution=True)
write_p = Task(description=f"Write report on {topic}", expected_output="Full report", agent=writer, context=[bg, trends])
review_p = Task(description="Review, return FINAL report", expected_output="Final Markdown", agent=reviewer, context=[write_p])

parallel_crew = Crew(agents=[researcher, writer, reviewer], tasks=[bg, trends, write_p, review_p],
                     process=Process.sequential)
print([t.async_execution for t in parallel_crew.tasks])

## 6. Hierarchical process

`Process.hierarchical` আর `manager_llm` দিলে একজন manager কাজ ভাগ করে ও ফল যাচাই করে। Manager-কে agents list-এ দেবেন না। ছোট model manager হিসেবে loop করতে পারে, তাই 8b বা তার বেশি ব্যবহার করুন।

In [ ]:
hier = Crew(agents=[researcher, writer, reviewer], tasks=[research, write, review],
            process=Process.hierarchical, manager_llm=llm)
print(hier.process)

## 7. Ollama check এবং (ঐচ্ছিক) run

Ollama বন্ধ থাকলে এই cell crash করে না, শুধু জানায়। চালু থাকলে report `report.md` ফাইলে লেখা হয়।

In [ ]:
import json, urllib.request, pathlib

def ollama_has(model="llama3.1:8b", base="http://localhost:11434"):
    try:
        with urllib.request.urlopen(f"{base}/api/tags", timeout=2) as r:
            return any(m["name"].startswith(model) for m in json.load(r)["models"])
    except Exception as e:
        print("Ollama নেই:", e)
        return False

if ollama_has():
    result = crew.kickoff()
    pathlib.Path("report.md").write_text(result.raw, encoding="utf-8")
    print(result.raw[:500])
else:
    print("Skip: `ollama serve` আর `ollama pull llama3.1:8b` চালান।")

## Achievement

পূর্ণ project: `../Class 2 Project/` — `python app.py "Topic" --mode sequential|parallel|hierarchical`। তিনটা mode-ই `pytest` দিয়ে Ollama ছাড়া test করা হয়।

## Exercises

1. Reviewer-এর backstory বদলে output তুলনা করুন।
2. `llama3.2:3b` দিয়ে চালান, মানের পার্থক্য লিখুন।
3. চতুর্থ agent "Fact Checker" যোগ করুন।